# imports and config

In [ ]:
from agno.agent import Agent
from agno.models.ollama import Ollama
from agno.knowledge.pdf_url import PDFUrlKnowledgeBase
from agno.vectordb.lancedb.lance_db import LanceDb
from agno.models.message import Message
from agno.embedder.sentence_transformer import SentenceTransformerEmbedder
from agno.vectordb.search import SearchType
from agno.document.chunking.fixed import FixedSizeChunking
from agno.tools.duckduckgo import DuckDuckGoTools
from agno.tools import Toolkit
from agno.utils.log import logger
from agno.utils.pprint import pprint_run_response
from agno.agent import Agent
from agno.tools.thinking import ThinkingTools
from agno.tools.yfinance import YFinanceTools


from mem0 import MemoryClient


from typing import List


from ollama import chat


from transformers import pipeline


import os
from dotenv import load_dotenv
load_dotenv(".env")

True

# $P(w_5 \mid w_0, w_1, w_2, w_3, w_4)$


In [ ]:
generator = pipeline('text-generation', model="facebook/opt-125m")
response = generator("What is the captial of Himachal Pradesh?", max_length=100)

print("---")
print(response[0]["generated_text"])

Device set to use cuda:0
Truncation was not explicitly activated but `max_length` is provided a specific value, please use `truncation=True` to explicitly truncate examples to max length. Defaulting to 'longest_first' truncation strategy. If you encode pairs of sequences (GLUE-style) with the tokenizer you can select this strategy more precisely by providing a specific strategy to `truncation`.


---
What is the captial of Himachal Pradesh?

The Himachal Pradesh is a state of India. It is located in the state of Himachal Pradesh. It is a part of the Union Territory of Himachal Pradesh.

The Himachal Pradesh is a state of India. It is a part of the Union Territory of Himachal Pradesh. It is a part of the Union Territory of Himachal Pradesh. It is a part of the Union Territory


In [ ]:
generator = pipeline('text-generation', model="facebook/opt-125m")
response = generator("max has 25 points, norris has 18 points. so the person with more points is", max_length=200)

print("---")
print(response[0]["generated_text"])

Device set to use cuda:0
Truncation was not explicitly activated but `max_length` is provided a specific value, please use `truncation=True` to explicitly truncate examples to max length. Defaulting to 'longest_first' truncation strategy. If you encode pairs of sequences (GLUE-style) with the tokenizer you can select this strategy more precisely by providing a specific strategy to `truncation`.


---
max has 25 points, norris has 18 points. so the person with more points is the one with the most points.
I think that's the point.                                                                                                                                                                      


![https://openai.com/index/instruction-following/](https://images.ctfassets.net/kftzwdyauwt9/690sZjaYwbes9IMEgfyJ7O/f536896b496b13749518f53d4caaa713/Methods_Diagram_dark_mode.jpg?w=3840&q=80&fm=webp)
> Credit: https://openai.com/index/instruction-following/

In [ ]:
response = chat(
    model='llama3.1',
    messages=[
        {
            'role': 'system',
            'content': 'You are a helpful assistant. Give succint, and concise one sentence answers. Always respond in hindi.',
        },
        {
            'role': 'user',
            'content': 'What is the captial of Himachal Pradesh?',
        },
    ]
)
print("---")
print(response.message.content)

---
शिमला हिमाचल प्रदेश की राजधानी है। (Shimla hai Himachal Pradesh ki rajdhani hai.)


In [ ]:
response = chat(
    model='llama3.1',
    messages=[
        {
            'role': 'user',
            'content': 'How many r are there in kookaburra?',
        },
    ]
)
print("---")
print(response.message.content)

---
There is one "r" in the word "kookaburra".


# 💡 Chain of Thought
A series of intermediate reasoning steps.

![Chain of Thought](https://www.promptingguide.ai/_next/image?url=%2F_next%2Fstatic%2Fmedia%2Fcot.1933d9fe.png&w=1920&q=75)
> Credit: https://arxiv.org/abs/2201.11903

In [ ]:
response = chat(
    model='llama3.1',
    messages=[
        {
            'role': 'user',
            'content': 'How many r are there in kookaburra? Go through each letter and check if it is r or not.',
        },
    ]
)
print("---")
print(response.message.content)

---
Let's go through each letter of "kookaburra":

1. K - Not an R
2. O - Not an R
3. O - Not an R
4. K - Not an R
5. A - Not an R
6. B - Not an R
7. U - Not an R
8. R - It is an R!
9. R - It is another R!
10. A - Not an R

So, there are 2 Rs in "kookaburra".


# 😔 Limitations of (Isolated) LLMs

### 😵‍💫 Hallucination: Generation of incorrect information with high confidence
### 🤨 Knowledge cutoff: Limited to training data timeframe
### 🥸 Lack of attribution: No direct source citations
### 🔒 Data privacy: Limited to public training data, no access to proprietary information
### ↔️ Limited context length: Constraints due to attention mechanism architecture


# 💡 RAG (Retrieval Augmented Generation)

![RAG Workflow](https://docs.aws.amazon.com/images/sagemaker/latest/dg/images/jumpstart/jumpstart-fm-rag.jpg)
> Credit: https://aws.amazon.com/what-is/retrieval-augmented-generation

In [ ]:
embedding_model = SentenceTransformerEmbedder()

vector_db = LanceDb(
    table_name="nvidia",
    embedder=embedding_model,
    search_type=SearchType.hybrid,
)
knowledge_base = PDFUrlKnowledgeBase(
    urls=["https://d18rn0p25nwr6d.cloudfront.net/CIK-0001045810/337ceb2e-b968-4d3f-8af3-a2853a66a0b5.pdf"],
    vector_db=vector_db,
    chunking_strategy=FixedSizeChunking(chunk_size=500, overlap=50),
    num_documents=3
)
knowledge_base.load()

agent = Agent(
    model=Ollama("llama3.2"),
    knowledge=knowledge_base,
    add_references=False,
    search_knowledge=True,
    show_tool_calls=True,
    markdown=True,
    debug_mode=False
)
response = agent.run(
    Message(
        role="user",
        content="What is project GR00T AI?",
    ),
)
print(response.content)

INFO Loading knowledge base

INFO Reading: https://d18rn0p25nwr6d.cloudfront.net/CIK-0001045810/337ceb2e-b968-4d3f-8af3-a2853a66a0b5.pdf

INFO No documents to insert

INFO Added 0 documents to knowledge base

Project GR00T AI is a project introduced by NVIDIA that aims to enhance production delivery through infrastructure as AI. It focuses on automotive and robotics, particularly on humanoid development, robot learning, and perception workflows for robotics developers. The project also includes the release of new generative AI tools and simulation tools for robot learning. Additionally, it has been announced that Japanese and Indian companies will be adopting agentic AI to revolutionize their workflows.


# 🤔 Knowledge Base
- A knowledge base is a database of information that an agent can search to **improve its responses**.
- This information is stored in a **vector database** and provides agents with business context, helping them respond in a context-aware manner.
- If you have **proprietary data**, you can use a knowledge base to provide agents with access to that data.
- If the LLM you are using has a **knowledge cutoff**, you can use a knowledge base to provide agents with access to the most up-to-date information.

![Hybrid Search](https://www.elastic.co/search-labs/_next/image?url=https%3A%2F%2Fcdn.sanity.io%2Fimages%2Fme0ej585%2Fsearch-labs-import-testing%2F31bf67d5450df5687e0e92b8fa617a2c675a80e8-1600x714.png&w=1200&q=75)
> Credit: https://www.elastic.co/search-labs/blog/hybrid-search-multiple-embeddings

![RAG Workflow with our classes](resources/cse641-llm-agents.drawio.png)

# 🛠️ Tool Use

In [ ]:
agent = Agent(tools=[DuckDuckGoTools()], show_tool_calls=True, model=Ollama("llama3.2"))
print("---")
agent.print_response("What is project GR00T AI?", markdown=True)

DEBUG Function: duckduckgo_search registered with duckduckgo                    
DEBUG Function: duckduckgo_news registered with duckduckgo                      
---
┏━ Message ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃                                                                              ┃
┃ What is project GR00T AI?                                                    ┃
┃                                                                              ┃
┗━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┛
┏━ Tool Calls ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃                                                                              ┃
┃ • duckduckgo_search(query=project GR00T AI)                                  ┃
┃                                                                              ┃
┗━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┛
┏━ Response (3.6s) ━━━━━

In [ ]:
class ShellTools(Toolkit):
    def __init__(self):
        super().__init__(name="shell_tools")
        self.register(self.run_shell_command)

    def run_shell_command(self, args: List[str], tail: int = 100) -> str:
        """
        Runs a shell command and returns the output or error.

        Args:
            args (List[str]): The command to run as a list of strings.
            tail (int): The number of lines to return from the output.
        Returns:
            str: The output of the command.
        """
        import subprocess

        logger.info(f"Running shell command: {args}")
        try:
            logger.info(f"Running shell command: {args}")
            result = subprocess.run(args, capture_output=True, text=True)
            logger.debug(f"Result: {result}")
            logger.debug(f"Return code: {result.returncode}")
            if result.returncode != 0:
                return f"Error: {result.stderr}"
            # return only the last n lines of the output
            return "\n".join(result.stdout.split("\n")[-tail:])
        except Exception as e:
            logger.warning(f"Failed to run shell command: {e}")
            return f"Error: {e}"

agent = Agent(tools=[ShellTools()], show_tool_calls=True, markdown=True, model=Ollama("llama3.1"))
# agent.print_response("List all the files in /home/puneet/agents/ directory with their permissions. And then express it in human readable format.")
response = agent.run(
    Message(
        role="user",
        content="List all the files in /home/puneet/agents/ directory with their permissions",
    ),
)
print(response.content)

INFO Running shell command: ['ls', '-l']

INFO Running shell command: ['ls', '-l']

The `ls` command is used to list the files and directories in a specified location. The `-l` option is used to display detailed information about each file, including its permissions.

In this case, we have four items:

1. `llm-progression.ipynb`: This is a Jupyter Notebook file, which has read and write permissions for the owner (`puneet`) and group (`puneet`), but only read permission for others.
2. `README.md`: This is a Markdown file, which also has read and write permissions for the owner and group, but only read permission for others.
3. `resources`: This is a directory with read and execute permissions for the owner and group, but only read and execute permission for others.
4. `tmp`: This is another directory with read and execute permissions for the owner and group, but only read and execute permission for others.

Note that these permissions are specific to the owner (`puneet`) and group (`puneet`), which means that other users on the system will not have access to these file

# 🧠 Memory

In [ ]:
client = MemoryClient(os.getenv("MEM0_API_KEY"))

user_id = "agno"
messages = [
    {"role": "user", "content": "My name is Puneet. You are my calendar assistant."},
    {"role": "user", "content": "I live in Chandigarh."},
    {"role": "user", "content": "I'm going to a concert on April 8 from 6 - 9 PM. Mark it."},
]
# Comment out the following line after running the script once
client.add(messages, user_id=user_id)

agent = Agent(
    model=Ollama(id="llama3.1"),
    context={"memory": client.get_all(user_id=user_id)},
    add_context=True,
    instructions=[
        "You are my calendar assistant.",
        "You have to keep track of my schedule.",
        "When I ask you to check my schedule for a specific date and time, you should check if I have any events scheduled for that date and time.",
        "If I have an event scheduled, you should tell me that I have an event scheduled for that date and time.",
        "If I don't have an event scheduled, you should tell me that my schedule is free for that date and time.",
    ]
)
response = agent.run("I want to go to the movies on April 8 around 6:30 PM, is my schedule free?")

pprint_run_response(response)

messages = [{"role": i.role, "content": str(i.content)} for i in (response.messages or [])]
client.add(messages, user_id=user_id)

# https://app.mem0.ai/dashboard/memories

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Let me check your schedule for April 8 around 6:30 PM.                                                          │
│                                                                                                                 │
│ You have an event scheduled for that date and time. You are going to the movies on April 8 at 6:30 PM (based on │
│ a memory entry indicating you want to go to the movies on that day). However, I also noticed another event - a  │
│ concert from 6 PM to 9 PM. So, it seems like your schedule is already booked for the evening of April 8.        │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

{'results': []}

# 🗣️ Reasoning

1. **Reasoning Models**  
   - Specialized models trained to think before responding, ideal for single-shot tasks.  
   - Examples: OpenAI o-series, Claude 3.7 Sonnet, DeepSeek-R1.

2. **Reasoning Tools**  
   - Add structured thinking to non-reasoning models using tools like `ThinkingTools`.  
   - Improve accuracy by validating and iterating over results.

3. **Reasoning Agents**  
   - Combine chain-of-thought reasoning with tool use for complex tasks.  
   - Solve, validate, and iterate before generating the final response.

In [ ]:
thinking_agent = Agent(
    model=Ollama("llama3.1"),
    tools=[
        ThinkingTools(add_instructions=True),
        YFinanceTools(
            stock_price=True,
            analyst_recommendations=True,
            company_info=True,
            company_news=True,
        ),
    ],
    instructions="Use tables where possible",
    show_tool_calls=True,
    markdown=True,
    debug_mode=False
)
thinking_agent.print_response("Write a report comparing NVDA to TSLA", stream=True)

DEBUG Function: think registered with thinking_tools

DEBUG Function: get_current_stock_price registered with yfinance_tools

DEBUG Function: get_company_info registered with yfinance_tools

DEBUG Function: get_analyst_recommendations registered with yfinance_tools

DEBUG Function: get_company_news registered with yfinance_tools

Output()

/home/puneet/miniconda3/envs/resume-agent/lib/python3.11/site-packages/ollama/_types.py:81: 
PydanticDeprecatedSince211: Accessing the 'model_fields' attribute on the instance is deprecated. Instead, you 
should access this attribute from the model class. Deprecated in Pydantic V2.11 to be removed in V3.0.
  if key in self.model_fields:

/home/puneet/miniconda3/envs/resume-agent/lib/python3.11/site-packages/ollama/_types.py:82: 
PydanticDeprecatedSince211: Accessing the 'model_fields' attribute on the instance is deprecated. Instead, you 
should access this attribute from the model class. Deprecated in Pydantic V2.11 to be removed in V3.0.
  return self.model_fields[key].default is not None

In [ ]:
thinking_agent = Agent(
    model=Ollama("llama3.1"),
    tools=[
        ThinkingTools(add_instructions=True),
        YFinanceTools(
            stock_price=True,
            analyst_recommendations=True,
            company_info=True,
            company_news=True,
        ),
        DuckDuckGoTools(),
    ],
    instructions="You are a hedge fund analyst. Find relevant news articles and analyse them. Analyse the stock movement of S&P 500 and NASDAQ.",
    show_tool_calls=True,
    markdown=True,
    debug_mode=False
)
thinking_agent.print_response("What caused the market crash of 2025?", stream=True)

Output()

In [ ]:
thinking_agent = Agent(
    model=Ollama("llama3.1"),
    tools=[
        ThinkingTools(add_instructions=True),
        YFinanceTools(
            stock_price=True,
            analyst_recommendations=True,
            company_info=True,
            company_news=True,
        ),
        DuckDuckGoTools(),
    ],
    instructions="You are a hedge fund analyst. Find relevant news articles on the matter and analyse them. Analyse the stock movement of S&P 500 and NASDAQ. Talk in informal finance bro lingo.",
    show_tool_calls=True,
    markdown=True,
    debug_mode=False
)
thinking_agent.print_response("buy the dip? 👀", stream=True)

Output()

# 🤖 Agentic LLMs / LLM Agents / Agentic AI

Instead of a rigid binary definition, let’s think of Agents in terms of agency and autonomy.

- Level 0: Agents with no tools (basic inference tasks).
- Level 1: Agents with tools for autonomous task execution.
- Level 2: Agents with knowledge, combining memory and reasoning.
- Level 3: Teams of specialized agents collaborating on complex workflows.

> Credit: https://docs.agno.com/agents/introduction

# Anthropic's Take on Agents

- Workflows are systems where LLMs and tools are **orchestrated** through predefined code paths.
- Agents, on the other hand, are systems where **LLMs dynamically direct their own processes and tool usage**, maintaining control over how they accomplish tasks.


## Augnmented LLM (a workflow)
![anthropic-augmented-llm](https://www.anthropic.com/_next/image?url=https%3A%2F%2Fwww-cdn.anthropic.com%2Fimages%2F4zrzovbb%2Fwebsite%2Fd3083d3f40bb2b6f477901cc9a240738d3dd1371-2401x1000.png&w=3840&q=75)
> Credit: https://www.anthropic.com/engineering/building-effective-agents

## Parallel LLM Calls (again, a workflow)
![anthropic-parallel](https://www.anthropic.com/_next/image?url=https%3A%2F%2Fwww-cdn.anthropic.com%2Fimages%2F4zrzovbb%2Fwebsite%2F406bb032ca007fd1624f261af717d70e6ca86286-2401x1000.png&w=3840&q=75)
> Credit: https://www.anthropic.com/engineering/building-effective-agents

## Orchestrator (still a workflow)
![anthropic-orchestrator](https://www.anthropic.com/_next/image?url=https%3A%2F%2Fwww-cdn.anthropic.com%2Fimages%2F4zrzovbb%2Fwebsite%2F8985fc683fae4780fb34eab1365ab78c7e51bc8e-2401x1000.png&w=3840&q=75)
> Credit: https://www.anthropic.com/engineering/building-effective-agents

## Agent (finally!)
![anthropic-agent](https://www.anthropic.com/_next/image?url=https%3A%2F%2Fwww-cdn.anthropic.com%2Fimages%2F4zrzovbb%2Fwebsite%2F58d9f10c985c4eb5d53798dea315f7bb5ab6249e-2401x1000.png&w=3840&q=75)
> Credit: https://www.anthropic.com/engineering/building-effective-agents

# References
- Welcome to Agno - Agno - https://docs.agno.com
- Building Effective AI Agents - https://www.anthropic.com/engineering/building-effective-agents
- Hybrid Search with Multiple Embeddings - Elasticsearch Labs - https://www.elastic.co/search-labs/blog/hybrid-search-multiple-embeddings
- What is RAG? - Retrieval-Augmented Generation AI Explained - AWS - https://aws.amazon.com/what-is/retrieval-augmented-generation

# Tools Used:
- Agno - https://docs.agno.com
- Llama - https://llama.com
- Ollama - https://ollama.com
- Mem0 - https://mem0.ai

# Further Reading
- Aligning Language Models to Follow Instructions - https://openai.com/index/instruction-following/
- Training Language Models to Follow Instructions with Human Feedback - https://arxiv.org/abs/2203.02155
- Chain-of-Thought Prompting Elicits Reasoning in Large Language Models - https://arxiv.org/abs/2201.11903